# 08 · Monitoring

**Role of this layer.** Every stage of every run (and every replay) appends one
JSON object to `runs/events.jsonl`. The log is append-only, so it doubles as an
audit trail and could be shipped to a log platform as is.

The Monitoring page reads two sources, and the functions below (the ones behind
the page) turn them into numbers:
- the **run records**, one per run in its final state: activity, human review,
  tokens and cost, latency and failures;
- the **event log**, for what can happen several times in a run: guardrail
  firings, capability executions and replays.

To make sure there is something to show, this notebook first runs the six example
workflows in mock mode (approving only 1 and 2, as `cli demo` does).

In [1]:
import pandas as pd

from quantlab.config import load_settings

pd.set_option("display.max_colwidth", 110)
pd.set_option("display.width", 200)
settings = load_settings()

In [2]:
from quantlab.engine import run_request
from quantlab.mock_scenarios import SCENARIOS

for scenario in SCENARIOS:
    session = run_request(
        scenario.request, list(scenario.dataset_ids), "mock", approve=scenario.auto_approve
    )
    print(scenario.id, session.status)

chf_safe_haven_success completed
spread_frequency_mismatch_block blocked
arima_forecast_block blocked
prompt_injection_unknown_capability blocked


vix_full_sample_lookahead completed


buy_the_dip_tech completed


In [3]:
from quantlab.lineage import load_runs
from quantlab.monitoring import activity, monitored_runs, read_events

records = load_runs(settings.runs_dir)
events = read_events(settings.events_path)
counted = monitored_runs(records)
print(len(records), "run records,", len(counted), "counted by the page,", len(events), "events")
pd.Series(activity(records))

142 run records, 27 counted by the page, 1623 events


total              142.000000
completed           55.000000
blocked             53.000000
failed               0.000000
rejected             3.000000
completed_share      0.387324
blocked_share        0.373239
failed_share         0.000000
rejected_share       0.021127
dtype: float64

The page counts only the decided runs started on the Analysis page
(`monitored_runs`, printed above), and replays from every source: these headless runs
are not among them. To show what each function computes, the cells below apply them
to every run record.

In [4]:
from quantlab.monitoring import (
    grounded_narratives,
    planner_refusals,
    replay_results,
    review_outcomes,
)

pd.Series(
    {
        **review_outcomes(records),
        **replay_results(events),
        **grounded_narratives(records),
        **planner_refusals(records),
    }
)

approved             52.000000
rejected              3.000000
approval_rate         0.945455
edited                6.000000
edit_rate             0.115385
approval_median_s     0.035000
approval_p95_s       79.698750
replays               7.000000
matching              7.000000
replay_match_rate     1.000000
narratives           55.000000
grounded             55.000000
grounded_rate         1.000000
refused              31.000000
refused_share         0.218310
dtype: float64

Every guardrail and every capability, zeros included, most frequent first:

In [5]:
from quantlab.monitoring import capability_counts, guardrail_counts

pd.DataFrame(guardrail_counts(events))

,code,name,action,count
0,V10,MATERIAL_ASSUMPTION,APPROVE,252
1,V09,LOOK_AHEAD,APPROVE,40
2,P01,PLANNER_REFUSED,BLOCK,31
3,V08,FREQUENCY_MISMATCH,BLOCK,20
4,E04,ALIGNMENT_LOSS,WARN,18
5,V03,INVALID_PARAM,BLOCK,16
6,V02,UNKNOWN_CAPABILITY,BLOCK,10
7,R01,REQUEST_TOO_LONG,BLOCK,1
8,V01,INVALID_PLAN,BLOCK,0
9,V04,TYPE_MISMATCH,BLOCK,0


In [6]:
pd.DataFrame(capability_counts(events))

,capability,code,count
0,load_series,C01,69
1,resample,C02,52
2,flag_regime,C07,51
3,forward_return,C08,35
4,compare_groups,C09,35
5,hac_mean_difference,C10,35
6,compute_returns,C03,29
7,rolling_volatility,C06,25
8,spread,C05,6
9,difference,C04,0


Cost per run, from the tokens the API reported and the prices in
`config.LLM_PRICES_PER_MILLION_TOKENS`. Mock runs call no LLM and are left out, so
in mock mode the table has no runs and the cache saved nothing:

In [7]:
from quantlab.config import LLM_PRICES_PER_MILLION_TOKENS
from quantlab.monitoring import cache_savings, cost_table

display(pd.DataFrame(cost_table(records, LLM_PRICES_PER_MILLION_TOKENS)))
pd.Series(cache_savings(records, LLM_PRICES_PER_MILLION_TOKENS))

,stage,runs,input_mean,input_p95,output_mean,output_p95,cost_mean,cost_p95,cost_total
0,planner,28,3833.035714,5004.0,378.464286,891.45,0.000573,0.000906,0.016031
1,narrator,28,411.750000,1578.0,50.178571,168.15,0.000066,0.000231,0.001855
2,total,28,4244.785714,5958.0,428.642857,955.25,0.000639,0.001051,0.017886


planner_calls             28.000000
cache_hits                 4.000000
hit_rate                   0.142857
saved_input_tokens     17504.000000
saved_output_tokens     2985.000000
saved_cost                 0.003243
dtype: float64

Latency per stage (mean and 95th percentile, milliseconds), and the most recent
failures with the stage and the guardrail that stopped them:

In [8]:
from quantlab.monitoring import latency_table, recent_failures

pd.DataFrame(latency_table(records))

,stage,runs,mean_ms,p95_ms,recorded
0,Request check,0,NaN,NaN,False
1,Planner,141,975.163121,7356.700,True
2,Validator,141,0.142624,0.340,True
3,Executor,55,122.106909,201.631,True
4,Narrator,55,530.896364,3010.170,True


In [9]:
pd.DataFrame(recent_failures(records, limit=5))

,time,run_id,status,stage,code,message
0,2026-10-19T08:30:51.494Z,20261019-083051-575957,blocked,Validator,V02,The plan has blocking issues: V02:s2
1,2026-10-19T08:30:51.468Z,20261019-083051-787961,blocked,Planner,P01,The plan has blocking issues: P01:plan
2,2026-10-19T08:30:51.439Z,20261019-083051-567cb5,blocked,Validator,V08,The plan has blocking issues: V08:s4
3,2026-10-19T08:30:16.191Z,20261019-083016-5550bc,blocked,Planner,P01,The plan has blocking issues: P01:plan
4,2026-10-19T08:30:00.887Z,20261019-083000-02dba4,blocked,Planner,P01,The plan has blocking issues: P01:plan
